# Notebook 1 de 3 — Fotos reales de autos

Proyecto **Detección de fraude**. Empezamos el dataset desde cero, con una idea de diseño: **las tres clases tienen que salir de la misma "fábrica"**. Lo único que puede cambiar entre una foto auténtica y una falsa es la manipulación, no el tamaño, la calidad ni el origen del archivo. Así el modelo no puede hacer trampa.

**El plan completo (3 notebooks):**

1. **Este notebook:** conseguir fotos reales de autos (sanos y con daño real) y dejarlas todas con el mismo formato base: 768×512 px, un solo auto bien visible, sin duplicados.
2. **Notebook 2:** a partir de esas fotos generar las falsas: (a) misma foto con daño agregado por IA (inpainting), (b) misma foto con una edición de *control* sin daño y (c) autos 100 % generados por IA.
3. **Notebook 3:** pasar todas las imágenes por el mismo paso final (misma compresión JPEG), chequear que no queden atajos y dejar el dataset listo para entrenar.

**Qué hace este notebook**

- Busca y descarga fotos de **Wikimedia Commons** (fotos con licencias abiertas) y guarda autor y licencia de cada una.
- Descarta automáticamente las que no sirven: sin auto claro, autos muy chicos en la foto, varios autos, duplicados.
- Recorta y redimensiona todo a 768×512 y calcula dónde está el auto (lo necesita el notebook 2).
- Te deja **revisar a ojo** las fotos y sacar las que no sirvan.
- Opcional: sumar **tus propias fotos de celular**, que son las más parecidas a un siniestro real.

**Cómo usarlo:** Colab, celdas de arriba hacia abajo. Con `PRUEBA_RAPIDA = True` baja muy pocas fotos para probar que todo funciona; después lo ponés en `False`. Conviene usar GPU (*Entorno de ejecución → Cambiar tipo → GPU*), el detector de autos anda más rápido.

> **Sobre las licencias:** las fotos de Commons son CC0, dominio público, CC BY o CC BY-SA. Para un proyecto de estudio interno alcanza con guardar autor y licencia (el notebook lo hace). Si algún día el dataset o el modelo se publican o se usan comercialmente, hay que revisar esas licencias con cuidado.

## 1. Instalación e imports

In [12]:
%pip install -q requests pillow pillow-heif pandas matplotlib

In [13]:
import os, sys, re, io, time, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
try:                                    # permite leer fotos HEIC de iPhone
    from pillow_heif import register_heif_opener
    register_heif_opener()
except Exception:
    pass

import torch
import torchvision
from torchvision.transforms import functional as TF

print("PyTorch", torch.__version__, "| torchvision", torchvision.__version__)

PyTorch 2.11.0+cu128 | torchvision 0.26.0+cu128


## 2. Configuración

Lo único que **tenés** que editar es `USER_AGENT` (Wikimedia pide que quien descarga se identifique con un mail de contacto). El resto son valores razonables.

- **Tamaño base 768×512** (proporción 3:2, como una foto de celular apaisada). Todo el proyecto usa este tamaño.
- Se piden a Commons **miniaturas de 1280 px de ancho** (más livianas que el original). Solo se aceptan fotos cuyo original tenga al menos 1600 px de ancho, para que al reducir queden nítidas.
- `CONSULTAS` son las búsquedas. Buscamos por **modelo de auto dentro del nombre del archivo** (`intitle:"Toyota Corolla"`), que es lo más preciso que ofrece Commons, y excluimos títulos con palabras como "car park", "railway" o "museum". Igual los resultados son irregulares: por eso después hay filtros automáticos y una revisión visual.
- **Fotos con daño real:** Commons tiene pocas fotos de autos dañados con calidad suficiente. Es normal que salgan pocas; las alternativas (fotos propias, CarDD) están en la sección 4 y al final.

In [14]:
# ---------- Dónde guardamos todo ----------
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ = Path("/content/drive/MyDrive/deteccion-fraude-v2")      # carpeta nueva, no toca lo anterior
else:
    RAIZ = Path("./deteccion-fraude-v2")

CARPETA          = RAIZ / "01_reales"
DIR_ORIGINALES   = CARPETA / "originales"       # miniaturas descargadas (1280 px)
DIR_BASE         = CARPETA / "base"             # fotos ya estandarizadas 768x512 (PNG sin pérdida)
DIR_PROPIAS      = CARPETA / "fotos_propias"    # opcional: subcarpetas "sanas" y "danadas"
for d in [DIR_ORIGINALES, DIR_BASE, DIR_PROPIAS / "sanas", DIR_PROPIAS / "danadas"]:
    d.mkdir(parents=True, exist_ok=True)
MANIFEST_DESCARGAS = CARPETA / "manifest_descargas.csv"
MANIFEST_REALES    = CARPETA / "manifest_reales.csv"

# ---------- Identificación para Wikimedia (EDITAR) ----------
USER_AGENT = "deteccion-fraude-estudio/1.0 (tu-mail@ejemplo.com)"   # <-- poné tu mail real (lo pide la política de uso de Wikimedia)

# ---------- Formato base ----------
CANVAS_W, CANVAS_H = 768, 512
ANCHO_DESCARGA     = 1280        # ancho de la miniatura que pedimos a Commons
MIN_ANCHO_ORIGINAL = 1600        # descartamos fotos originales más chicas

# ---------- Cuántas queremos ----------
OBJETIVO = {"sano": 300, "danado": 100}
MAX_POR_CONSULTA = 120           # tope de resultados por búsqueda
MAX_ACEPTADAS_POR_CONSULTA = 25  # tope de fotos que bajamos por búsqueda (así no sale todo del mismo modelo)

# ---------- Búsquedas en Commons ----------
# Buscamos por MODELO dentro del NOMBRE del archivo (intitle:): casi todas las fotos de autos en Commons se
# llaman "Toyota Corolla E210 front.jpg". Buscar palabras sueltas como "car street" trae calles y estacionamientos.
# Los "-intitle:..." excluyen títulos con esas palabras.
EXCLUIR = ('-intitle:interior -intitle:engine -intitle:museum -intitle:show -intitle:railway -intitle:train '
           '-intitle:toy -intitle:model -intitle:"car park" -intitle:parking')
MODELOS = ["Toyota Corolla", "Volkswagen Golf", "Ford Focus", "Honda Civic", "Renault Clio", "Peugeot 208",
           "Fiat Punto", "Volkswagen Gol", "Renault Sandero", "Chevrolet Onix", "Toyota Hilux", "Ford Fiesta",
           "Opel Astra", "Hyundai i20", "Nissan Micra", "Kia Rio", "Skoda Octavia", "Mazda 3"]
CONSULTAS = {
    "sano": [f'intitle:"{m}" {EXCLUIR}' for m in MODELOS],
    "danado": [f'intitle:"{t}" {EXCLUIR}' for t in
               ["car crash", "car accident", "damaged car", "crashed car", "car collision", "wrecked car",
                "hail damage", "dented", "traffic collision", "accident damage"]],
}
# Además, si el título contiene alguna de estas palabras, se descarta sin siquiera descargar la foto
TITULOS_EXCLUIDOS = (r"car park|parking|railway|train\b|tank car|interior|engine|museum|toy|lego|diecast|drawing|painting|logo|"
                     r"wheel|steering|dashboard|riot|unrest|protest|demonstration|bomb|explosion|war\b|attack|burnt|burned|fire\b")

# ---------- Empezar de cero ----------
# Poné True UNA vez si querés borrar lo descargado y procesado antes (no toca fotos_propias) y volver a bajar todo.
REINICIAR = False

# ---------- Filtros de calidad ----------
MIN_PUNTAJE_AUTO   = 0.70        # confianza mínima del detector para contar un auto
MIN_FRAC_AUTO      = 0.20        # el auto debe ocupar al menos 20 % de la foto
MIN_FRAC_EN_RECORTE = 0.80       # y quedar mayormente dentro del recorte 3:2 (un auto apenas cortado es realista)
UMBRAL_DUPLICADO   = 6           # distancia de hash: <= 6 se considera la misma foto

# ---------- Modo prueba ----------
PRUEBA_RAPIDA = False
if PRUEBA_RAPIDA:
    OBJETIVO = {"sano": 12, "danado": 6}          # bajamos de más: los filtros descartan cerca de la mitad
    MAX_POR_CONSULTA = 12
    MAX_ACEPTADAS_POR_CONSULTA = 3                # 3 fotos por modelo -> salen de 4 modelos distintos
    CONSULTAS = {"sano": CONSULTAS["sano"][:4], "danado": CONSULTAS["danado"][:4]}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA)
print("Dispositivo:", DEVICE, "| carpeta de trabajo:", CARPETA.resolve())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dispositivo: cuda | carpeta de trabajo: /content/drive/MyDrive/deteccion-fraude-v2/01_reales


## 3. Buscar y descargar de Wikimedia Commons

La API de Commons se puede consultar sin cuenta. Por cada archivo pedimos: dirección de descarga de la miniatura, tamaño original, tipo de archivo, **licencia** y **autor**. Nos quedamos solo con:

- fotos JPEG cuyo original tenga al menos `MIN_ANCHO_ORIGINAL` px de ancho,
- licencias que permiten hacer obras derivadas (CC0, dominio público, CC BY, CC BY-SA). Se descartan las **NC** (no comercial) y **ND** (no derivados): como vamos a editar las fotos con IA, las ND no se pueden usar.

Cada foto queda registrada en `manifest_descargas.csv`. Si la celda se corta o se reinicia Colab, al volver a ejecutarla **retoma** desde donde quedó, sin bajar dos veces la misma foto.

In [15]:
API = "https://commons.wikimedia.org/w/api.php"
SESION = requests.Session()
SESION.headers.update({"User-Agent": USER_AGENT})

def api_get(params, reintentos=5):
    # Consulta a la API de Commons con reintentos y espera si nos frenan (429).
    p = {"format": "json", "formatversion": "2", **params}
    for intento in range(reintentos):
        r = SESION.get(API, params=p, timeout=60)
        if r.status_code == 200:
            return r.json()
        time.sleep(2 * (intento + 1))
    r.raise_for_status()

def descargar_bytes(url, reintentos=5):
    for intento in range(reintentos):
        r = SESION.get(url, timeout=120)
        if r.status_code == 200:
            return r.content
        time.sleep(3 * (intento + 1))
    r.raise_for_status()

def licencia_ok(texto):
    t = (texto or "").lower().strip()
    partes = re.split(r"[\s\-_/]+", t)
    if "nc" in partes or "nd" in partes:
        return False
    return t.startswith(("cc by", "cc-by", "cc0", "public domain", "pd"))

def limpiar_html(texto):
    return re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", texto or "")).strip()

def buscar_commons(consulta, max_resultados):
    params = {
        "action": "query", "generator": "search",
        "gsrsearch": f"{consulta} filetype:bitmap", "gsrnamespace": 6, "gsrlimit": 50,
        "prop": "imageinfo", "iiprop": "url|size|mime|extmetadata", "iiurlwidth": ANCHO_DESCARGA,
        "iiextmetadatafilter": "LicenseShortName|Artist|LicenseUrl",
    }
    resultados = []
    while len(resultados) < max_resultados:
        resp = api_get(params)
        for pg in resp.get("query", {}).get("pages", []):
            info = (pg.get("imageinfo") or [None])[0]
            if info:
                resultados.append({"titulo": pg["title"], "info": info})
        if "continue" in resp:
            params.update(resp["continue"])
        else:
            break
    return resultados[:max_resultados]

def es_valida(info, titulo=""):
    meta = info.get("extmetadata", {})
    lic = meta.get("LicenseShortName", {}).get("value", "")
    if re.search(TITULOS_EXCLUIDOS, titulo, flags=re.IGNORECASE):
        return False
    return (info.get("mime") == "image/jpeg"
            and info.get("width", 0) >= MIN_ANCHO_ORIGINAL
            and bool(info.get("thumburl"))
            and licencia_ok(lic))

In [16]:
if REINICIAR:
    for p in list(DIR_ORIGINALES.glob("*")) + list(DIR_BASE.glob("*")):
        p.unlink()
    for p in [MANIFEST_DESCARGAS, MANIFEST_REALES]:
        if p.exists():
            p.unlink()
    print("REINICIAR = True: se borró lo descargado y procesado antes (fotos_propias no se toca).")
    print("Acordate de volver a poner REINICIAR = False para no borrar de nuevo si repetís esta celda.\n")

COLUMNAS_DESC = ["id", "tipo", "fuente", "titulo", "url_pagina", "autor", "licencia",
                 "ancho_orig", "alto_orig", "consulta", "ruta_local"]

if MANIFEST_DESCARGAS.exists():
    df_desc = pd.read_csv(MANIFEST_DESCARGAS)
else:
    df_desc = pd.DataFrame(columns=COLUMNAS_DESC)

titulos_vistos = set(df_desc["titulo"].dropna())
contador = {"sano": 0, "danado": 0}
for _, f in df_desc[df_desc["fuente"] == "commons"].iterrows():
    contador[f["tipo"]] += 1

def nuevo_id(tipo):
    prefijo = "r" if tipo == "sano" else "d"
    return f"{prefijo}{contador[tipo] + 1:04d}"

nuevas = []
for tipo, consultas in CONSULTAS.items():
    for consulta in consultas:
        if contador[tipo] >= OBJETIVO[tipo]:
            break
        print(f"[{tipo}] buscando: {consulta!r}")
        de_esta_consulta = 0
        for res in buscar_commons(consulta, MAX_POR_CONSULTA):
            if contador[tipo] >= OBJETIVO[tipo] or de_esta_consulta >= MAX_ACEPTADAS_POR_CONSULTA:
                break
            info, titulo = res["info"], res["titulo"]
            if titulo in titulos_vistos or not es_valida(info, titulo):
                continue
            try:
                datos = descargar_bytes(info["thumburl"])
            except Exception as e:
                print("   no se pudo bajar", titulo, e)
                continue
            id_ = nuevo_id(tipo)
            ruta = DIR_ORIGINALES / f"{id_}.jpg"
            ruta.write_bytes(datos)
            meta = info.get("extmetadata", {})
            fila = {"id": id_, "tipo": tipo, "fuente": "commons", "titulo": titulo,
                    "url_pagina": info.get("descriptionurl", ""),
                    "autor": limpiar_html(meta.get("Artist", {}).get("value", "")),
                    "licencia": meta.get("LicenseShortName", {}).get("value", ""),
                    "ancho_orig": info.get("width"), "alto_orig": info.get("height"),
                    "consulta": consulta, "ruta_local": str(ruta)}
            nuevas.append(fila)
            titulos_vistos.add(titulo)
            contador[tipo] += 1
            de_esta_consulta += 1
            time.sleep(0.5)            # somos amables con el servidor
        # guardamos después de cada búsqueda por si se corta
        if nuevas:
            df_desc = pd.concat([df_desc, pd.DataFrame(nuevas)], ignore_index=True)
            df_desc.to_csv(MANIFEST_DESCARGAS, index=False)
            nuevas = []

print("\nDescargadas hasta ahora:", contador)


Descargadas hasta ahora: {'sano': 300, 'danado': 100}


## 4. Tus propias fotos (opcional, muy recomendable)

Las fotos de celular de autos reales son lo más parecido a un reclamo de siniestro. Si podés, subí a Drive fotos de autos (estacionados en la calle, tomadas con tu celular o el de amigos, **con permiso** y evitando que se vea la patente si querés) en estas dos carpetas dentro de `deteccion-fraude-v2/01_reales/fotos_propias/`:

- `sanas/`: autos sin daño.
- `danadas/`: autos con daño real (si tenés alguna).

Con 50 a 100 fotos ya suman mucho. Pasan por exactamente los mismos filtros que las de Commons. Si no tenés, salteá esta celda: no pasa nada.

In [17]:
def registrar_fotos_propias(df):
    ya = set(df["ruta_local"].dropna())
    filas, cont = [], {"sano": 0, "danado": 0}
    for _, f in df[df["fuente"] == "propia"].iterrows():
        cont[f["tipo"]] += 1
    for tipo, sub in [("sano", "sanas"), ("danado", "danadas")]:
        for p in sorted((DIR_PROPIAS / sub).glob("*")):
            if p.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp", ".heic"} or str(p) in ya:
                continue
            cont[tipo] += 1
            pref = "rp" if tipo == "sano" else "dp"
            filas.append({"id": f"{pref}{cont[tipo]:03d}", "tipo": tipo, "fuente": "propia",
                          "titulo": p.name, "url_pagina": "", "autor": "propia", "licencia": "propia",
                          "ancho_orig": None, "alto_orig": None, "consulta": "", "ruta_local": str(p)})
    if filas:
        df = pd.concat([df, pd.DataFrame(filas)], ignore_index=True)
        df.to_csv(MANIFEST_DESCARGAS, index=False)
    return df, len(filas)

df_desc, n_propias = registrar_fotos_propias(df_desc)
print(f"Fotos propias nuevas registradas: {n_propias}")
print(df_desc.groupby(["fuente", "tipo"]).size().to_string())

Fotos propias nuevas registradas: 0
fuente   tipo  
commons  danado    100
         sano      300


## 5. Detector de autos y funciones de estandarización

Para saber **dónde está el auto** usamos un detector ya entrenado (Faster R-CNN de torchvision, entrenado con el dataset COCO, que incluye la clase "car"). Con eso:

- nos quedamos solo con fotos donde hay **un auto principal claro** (que ocupe una parte importante de la foto y sin otro auto casi igual de grande al lado);
- recortamos a proporción 3:2 **centrando en el auto**, y descartamos si el recorte lo corta demasiado;
- guardamos la **caja del auto** (`caja_x0…caja_y1`, ya en coordenadas de 768×512). El notebook 2 la usa para editar solamente zonas de la carrocería.

También sacamos **duplicados** con un "hash perceptual" (una huella de 64 bits de la imagen: dos fotos casi iguales tienen huellas casi iguales).

In [18]:
# [CARGA_MODELO] Detector de autos (se descarga una vez, ~160 MB)
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights

_PESOS = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
detector = fasterrcnn_resnet50_fpn_v2(weights=_PESOS, box_score_thresh=0.5).eval().to(DEVICE)
ID_AUTO = _PESOS.meta["categories"].index("car")
ID_CAMION = _PESOS.meta["categories"].index("truck")

@torch.no_grad()
def detectar_autos(img):
    # Devuelve lista de cajas [x0, y0, x1, y1] de autos/camionetas con buena confianza.
    t = TF.to_tensor(img).to(DEVICE)
    out = detector([t])[0]
    cajas = []
    for b, l, s in zip(out["boxes"].cpu().numpy(), out["labels"].cpu().numpy(), out["scores"].cpu().numpy()):
        if int(l) in (ID_AUTO, ID_CAMION) and float(s) >= MIN_PUNTAJE_AUTO:
            cajas.append(b)
    return cajas

In [19]:
def area(b):
    return max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])

def auto_principal(img):
    # Devuelve la caja del auto principal o (None, motivo) si la foto no sirve.
    cajas = detectar_autos(img)
    if not cajas:
        return None, "sin_auto"
    areas = [area(b) for b in cajas]
    orden = np.argsort(areas)[::-1]
    principal = cajas[orden[0]]
    if areas[orden[0]] / (img.width * img.height) < MIN_FRAC_AUTO:
        return None, "auto_muy_chico"
    if len(cajas) > 1 and areas[orden[1]] > 0.5 * areas[orden[0]]:
        return None, "varios_autos"
    return principal, "ok"

def recorte_3_2(img, caja):
    # Ventana 3:2 lo más grande posible, centrada en el auto. Devuelve (ventana, fracción del auto adentro).
    W, H = img.size
    asp = CANVAS_W / CANVAS_H
    if W / H >= asp:
        ch, cw = H, int(round(H * asp))
    else:
        cw, ch = W, int(round(W / asp))
    cx, cy = (caja[0] + caja[2]) / 2, (caja[1] + caja[3]) / 2
    x0 = int(round(min(max(cx - cw / 2, 0), W - cw)))
    y0 = int(round(min(max(cy - ch / 2, 0), H - ch)))
    ventana = (x0, y0, x0 + cw, y0 + ch)
    inter = area((max(caja[0], ventana[0]), max(caja[1], ventana[1]),
                  min(caja[2], ventana[2]), min(caja[3], ventana[3])))
    return ventana, inter / max(area(caja), 1e-6)

def dhash(img, n=8):
    g = img.convert("L").resize((n + 1, n), Image.LANCZOS)
    a = np.asarray(g, dtype=np.int16)
    return (a[:, 1:] > a[:, :-1]).flatten()

def estandarizar(ruta):
    # Devuelve (imagen 768x512, caja en coords base, hash, motivo). Si no sirve: imagen=None.
    img = ImageOps.exif_transpose(Image.open(ruta)).convert("RGB")
    caja, motivo = auto_principal(img)
    if caja is None:
        return None, None, None, motivo
    ventana, frac = recorte_3_2(img, caja)
    if frac < MIN_FRAC_EN_RECORTE:
        return None, None, None, "auto_cortado"
    cw = ventana[2] - ventana[0]
    if cw < CANVAS_W:
        return None, None, None, "foto_chica"
    recorte = img.crop(ventana).resize((CANVAS_W, CANVAS_H), Image.LANCZOS)
    k = CANVAS_W / cw
    caja_base = [float(np.clip((caja[0] - ventana[0]) * k, 0, CANVAS_W)),
                 float(np.clip((caja[1] - ventana[1]) * k, 0, CANVAS_H)),
                 float(np.clip((caja[2] - ventana[0]) * k, 0, CANVAS_W)),
                 float(np.clip((caja[3] - ventana[1]) * k, 0, CANVAS_H))]
    return recorte, caja_base, dhash(recorte), "ok"

## 6. Procesar todas las fotos descargadas

Para cada foto: detectar el auto, recortar, redimensionar a 768×512 y guardar como **PNG** (sin pérdida, así no agregamos una compresión más; la compresión final la aplicamos una sola vez y igual para todas en el notebook 3). Las que no pasan los filtros quedan anotadas con el motivo, en la columna `estado`.

In [20]:
hashes_vistos = []            # huellas de las fotos ya aceptadas
filas = []
for i, f in df_desc.iterrows():
    ruta_base = DIR_BASE / f"{f['id']}.png"
    try:
        img_b, caja, h, motivo = estandarizar(f["ruta_local"])
    except Exception as e:
        img_b, caja, h, motivo = None, None, None, f"error_lectura"
    if img_b is not None:
        if any(int((h != h2).sum()) <= UMBRAL_DUPLICADO for h2 in hashes_vistos):
            img_b, motivo = None, "duplicada"
        else:
            hashes_vistos.append(h)
            img_b.save(ruta_base)
    fila = f.to_dict()
    fila.update({"estado": motivo,
                 "caja_x0": caja[0] if caja else None, "caja_y0": caja[1] if caja else None,
                 "caja_x1": caja[2] if caja else None, "caja_y1": caja[3] if caja else None,
                 "ruta_base": str(ruta_base) if img_b is not None else ""})
    filas.append(fila)

df_proc = pd.DataFrame(filas)
print("Resultado del filtrado automático:")
print(pd.crosstab(df_proc["tipo"], df_proc["estado"]).to_string())
print("\nDetalle (primeras 40 filas):")
print(df_proc[["id", "tipo", "titulo", "estado"]].head(40).to_string())
if (df_proc["estado"] == "ok").sum() == 0:
    print("\nNinguna foto pasó el filtro. Mirá la columna 'estado': si casi todo es 'sin_auto' o 'auto_muy_chico', "
          "las búsquedas están trayendo escenas de calle y no autos; revisá CONSULTAS y TITULOS_EXCLUIDOS.")

Resultado del filtrado automático:
estado  auto_cortado  auto_muy_chico  duplicada   ok  sin_auto  varios_autos
tipo                                                                        
danado             1              32          1   38        23             5
sano              15              13          1  245         7            19

Detalle (primeras 40 filas):
       id    tipo                                                                           titulo          estado
0   r0001    sano                             File:2018 Volkswagen Golf R-Line TSi EVO S-A 1.5.jpg              ok
1   r0002    sano                                     File:2020 Volkswagen Golf Style 1.5 Side.jpg    auto_cortado
2   r0003    sano                                     File:2020 Volkswagen Golf Style 1.5 Rear.jpg    varios_autos
3   r0004    sano                                                  File:'05-'07 Ford Focus ZXW.jpg              ok
4   r0005    sano                              File:

## 7. Revisión visual (importante, tarda pocos minutos)

Los filtros automáticos no entienden todo. Mirá las fotos y anotá los `id` de las que **no sirven** (pinturas, maquetas, autos de carrera, fotos con mucha gente, mal etiquetadas como sanas/dañadas, etc.).

Las hojas de contacto muestran 30 fotos por vez con su `id` arriba. Esto es lo que más mejora la calidad del dataset: 10 minutos de mirar acá ahorran horas después.

In [21]:
def hoja_contacto(tipo, pagina=0, por_hoja=30, columnas=6):
    sub = df_proc[(df_proc["estado"] == "ok") & (df_proc["tipo"] == tipo)].reset_index(drop=True)
    chunk = sub.iloc[pagina * por_hoja:(pagina + 1) * por_hoja]
    filas_n = int(np.ceil(max(len(chunk), 1) / columnas))
    fig, ax = plt.subplots(filas_n, columnas, figsize=(columnas * 2.6, filas_n * 2.0))
    ax = np.atleast_1d(ax).flatten()
    for a in ax:
        a.axis("off")
    for a, (_, f) in zip(ax, chunk.iterrows()):
        a.imshow(Image.open(f["ruta_base"]))
        a.set_title(f["id"], fontsize=9)
    plt.suptitle(f"{tipo.upper()} - página {pagina} (de {int(np.ceil(len(sub) / por_hoja))})", y=1.0)
    plt.tight_layout(); plt.show()

hoja_contacto("sano", 0)

Output hidden; open in https://colab.research.google.com to view.

In [22]:
# Cambiá el número de página (1, 2, ...) para ver más. Si no hay dañadas todavía, no muestra nada.
hoja_contacto("danado", 0)

Output hidden; open in https://colab.research.google.com to view.

In [23]:
# Anotá acá los ids que NO sirven y las correcciones de tipo. Después volvé a ejecutar esta celda y las siguientes.
DESCARTAR = ["r0005"]
CORREGIR_TIPO = {
    # "r0012": "danado",      # estaba como sana pero tiene daño
}
print("A descartar:", DESCARTAR, "| a corregir:", CORREGIR_TIPO)

A descartar: ['r0005'] | a corregir: {}


## 8. Resultado final: `manifest_reales.csv`

Se guardan solo las fotos aceptadas y no descartadas. Este archivo es la entrada del notebook 2. Además dejamos un resumen y un archivo `atribuciones.csv` con autor, licencia y enlace de cada foto de Commons (para poder citarlas).

In [24]:
df_ok = df_proc[(df_proc["estado"] == "ok") & (~df_proc["id"].isin(DESCARTAR))].copy()
for id_, nuevo_tipo in CORREGIR_TIPO.items():
    df_ok.loc[df_ok["id"] == id_, "tipo"] = nuevo_tipo

columnas_final = ["id", "tipo", "fuente", "ruta_base", "caja_x0", "caja_y0", "caja_x1", "caja_y1",
                  "titulo", "url_pagina", "autor", "licencia"]
df_ok[columnas_final].to_csv(MANIFEST_REALES, index=False)
df_ok[df_ok["fuente"] == "commons"][["id", "titulo", "url_pagina", "autor", "licencia"]].to_csv(
    CARPETA / "atribuciones.csv", index=False)

print("Fotos reales aceptadas:")
print(df_ok.groupby(["fuente", "tipo"]).size().to_string())
print("\nLicencias:", df_ok["licencia"].value_counts().to_dict())

if (df_ok["tipo"] == "sano").sum() < 150:
    print("\nATENCIÓN: tenés menos de 150 fotos sanas. Para un buen entrenamiento conviene apuntar a 300 o más "
          "(subí OBJETIVO, agregá consultas o sumá fotos propias).")
if (df_ok["tipo"] == "danado").sum() < 50:
    print("ATENCIÓN: pocas fotos con daño real. Las necesitamos en la clase auténtica para que el modelo "
          "no aprenda 'hay un golpe = falso'.")
print("\nGuardado en:", MANIFEST_REALES)

Fotos reales aceptadas:
fuente   tipo  
commons  danado     38
         sano      244

Licencias: {'CC BY-SA 4.0': 134, 'CC BY-SA 2.0': 37, 'Public domain': 28, 'CC BY-SA 3.0': 24, 'CC BY 2.0': 18, 'CC BY 3.0': 11, 'CC BY 4.0': 10, 'CC0': 8, 'CC BY-SA 3.0 it': 4, 'CC BY-SA 3.0 de': 3, 'CC BY-SA 2.0 de': 2, 'CC BY-SA 3.0 gr': 1, 'CC BY 2.5': 1, 'CC BY-SA 2.5': 1}
ATENCIÓN: pocas fotos con daño real. Las necesitamos en la clase auténtica para que el modelo no aprenda 'hay un golpe = falso'.

Guardado en: /content/drive/MyDrive/deteccion-fraude-v2/01_reales/manifest_reales.csv


## 9. Qué sigue

- Si tenés menos fotos de las que querías, ampliá `MODELOS` con más marcas y modelos, poné `PRUEBA_RAPIDA = False` y volvé a ejecutar desde la sección 3: retoma sin repetir. Si cambiaste las búsquedas y querés arrancar limpio, poné `REINICIAR = True` una vez.
- Si salen pocas fotos **con daño real**, es esperable en Commons. Opciones: tus propias fotos en `fotos_propias/danadas/`, o pedir el dataset CarDD (4.000 fotos de alta resolución con daño real, requiere formulario de licencia) y copiar sus fotos a esa misma carpeta.
- Cuando el resumen te cierre, pasá al **notebook 2**, que toma `manifest_reales.csv` y genera las falsas.